# IIT Madras campus UTCI forecast

**Group 3 · Heat Adaptation for IITM Campus · v5**

Forecasts the 10:30 AM (Landsat overpass) Universal Thermal Climate Index for five campus zones, and updates itself as new scenes arrive from Google Earth Engine.

| Step | What happens |
|---|---|
| 1 | Load the raw GEE export, plus any scenes fetched since |
| 1b | *(optional)* Fetch new Landsat 8/9 scenes and ERA5-Land from Earth Engine |
| 1c | TVX downscaling to point-level air temperature, MRT, humidity and wind |
| 2–3 | Cloud QC, then UTCI per point with `pythermalcomfort`, averaged per zone |
| 4–5 | Regular 8-day grid; decomposition into climatology + trend + anomaly |
| 6–8 | Kalman AR(1) vs LSTM, scored by a rolling-origin replay *(full run only)* |
| 9 | Next-scene forecast, daily fill, forecast log and live scorecard |

**Run modes**

- **Full** (default): everything, including the replay and the LSTM. Takes a few minutes.
- **Quick** (`UTCI_MODE=quick`): fetch, refit Kalman, forecast and score. Replay metrics are kept from the last full run. This is what the scheduled GitHub refresh runs.

All settings are in **Configuration**. Environment variables override them, which is how the GitHub workflow drives the notebook.

### Data notes

- **Export format:** one row per point and scene; dates `dd-mm-yyyy`; zones `AC, HZ, LW, NV, SQ`. Cloudy pixels are dropped, so a scene has 1–500 points and fully cloudy scenes are absent.
- **Fetched scenes** are stored separately in `fetched_scenes.csv`, so the original export is never modified.
- **MRT:** in the export, `MeanRadiantTemp` equals `AirTemp_2m`. UTCI therefore has no solar term; only the LST deviation (`k_mrt`) adds radiant load.
- **Wind:** `gamma_wind = 0.6` was tuned on the earlier, compressed NDVI. With correctly scaled NDVI it cuts wind by up to about 60%.
- **Cloud contamination:** about 7% of points remain more than 5 °C below air temperature, so the QC rules stay on.

In [ ]:
import importlib, subprocess, sys

def ensure_pkg(stmt, pip_name):
    try:
        exec(stmt, {})
        return
    except Exception:
        pass
    print("Installing:", pip_name)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    importlib.invalidate_caches()
    top = stmt.split()[1].split(".")[0]
    for m in [m for m in sys.modules if m.split(".")[0] == top]:
        del sys.modules[m]

for stmt, pip in [("import pandas", "pandas"), ("import numpy", "numpy"), ("import scipy", "scipy"),
                  ("import statsmodels", "statsmodels"), ("import matplotlib", "matplotlib"),
                  ("from pythermalcomfort.models import utci", "pythermalcomfort>=3")]:
    ensure_pkg(stmt, pip)

import os, json, time, warnings, platform, zipfile
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from scipy.stats import linregress
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pythermalcomfort.models import utci as ptc_utci

warnings.filterwarnings("ignore")
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "statsmodels": statsmodels.__version__, "pythermalcomfort": pkg_version("pythermalcomfort")}
T_START = time.time()

PALETTE = {"maroon": "#7A1E2B", "gold": "#B8892B", "slate": "#4F6D8A", "ink": "#2B2F36",
           "sage": "#5F8A6E", "grey": "#8A8F98", "rule": "#E3DED6"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "savefig.bbox": "tight", "figure.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.edgecolor": PALETTE["grey"], "axes.labelcolor": PALETTE["ink"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": PALETTE["rule"],
    "grid.linewidth": 0.6, "xtick.color": PALETTE["ink"], "ytick.color": PALETTE["ink"], "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=[PALETTE[k] for k in ("maroon", "slate", "gold", "sage", "ink")]),
})
print(VERSIONS)

## Configuration

In [ ]:
ENV = os.environ.get
flag = lambda name, default: ENV(name, str(int(default))).strip().lower() in ("1", "true", "yes")

FULL_RUN = ENV("UTCI_MODE", "full").strip().lower() != "quick"
IN_CI = ENV("CI", "").lower() == "true"

DATA_PATH = ENV("UTCI_DATA_PATH", "IITM_GEE_5Zones_Microclimate_Complete_2013_2026.csv")
FETCHED_PATH = ENV("UTCI_FETCHED_PATH", os.path.join(os.path.dirname(DATA_PATH) or ".", "fetched_scenes.csv"))
OUT_DIR = ENV("UTCI_OUT_DIR", "outputs")
DATE_COL = "Observation_Date"

ZONE_NAMES = {"AC": "Academic Zone", "HZ": "Hostel Zone", "LW": "Lake + Watsa Ground",
              "NV": "Near Velachery Gate", "SQ": "Staff Quarters"}

DOWNSCALE = {"alpha_air": 0.15, "k_mrt": 0.5, "beta_rh": 5.0, "beta_vp": 0.05, "gamma_wind": 0.6}

RUN_GEE_FETCH = flag("UTCI_GEE_FETCH", True)
GEE_DRY_RUN = flag("UTCI_GEE_DRY_RUN", True)
GEE_PROJECT = ENV("GEE_PROJECT", "lofty-block-513811-h1")
GEE_MAX_NEW_SCENES = 40
GEE_FLAGS = {"fix_ndvi_scaling": True, "extra_cloud_mask": True, "fill_masked": False, "mrt_mode": "air"}

UTCI_INPUTS = {"tdb": "AirTemp_2m", "tr": "Local_MRT", "v": "Local_WindSpeed_10m", "rh": "Local_RelHumidity"}

GAPFILL_STD_TOL = 1e-6
LST_MIN_C, LST_MAX_C = 15.0, 70.0
LST_BELOW_AIR_C = 5.0
MIN_VALID_FRAC = 0.5

GRID_STEP_DAYS = 8
N_HARMONICS = 2
TEST_FRAC = 0.25
KALMAN_REFIT_EVERY = 1
PI_LEVEL = 0.90
FORECAST_MODEL = "kalman"

USE_LSTM = FULL_RUN
LSTM_WINDOW, LSTM_HIDDEN, LSTM_SEEDS = 8, 16, 3
LSTM_EPOCHS, LSTM_WARM_EPOCHS, LSTM_REFIT_EVERY = 80, 25, 4

os.makedirs(OUT_DIR, exist_ok=True)
LOG_PATH = os.path.join(OUT_DIR, "forecast_log.csv")
GEE_LOG_PATH = os.path.join(OUT_DIR, "gee_fetch_log.csv")
out = lambda name: os.path.join(OUT_DIR, name)

print(f"mode: {'full' if FULL_RUN else 'quick'} | GEE fetch: {RUN_GEE_FETCH} (dry run: {GEE_DRY_RUN}) | outputs: {OUT_DIR}")

<details><summary><b>Parameter reference</b></summary>

| Parameter | Meaning |
|---|---|
| `DOWNSCALE.alpha_air` | Share of the LST deviation passed to air temperature (TVX) |
| `DOWNSCALE.k_mrt` | MRT sensitivity to LST deviation from the zone-date mean |
| `DOWNSCALE.beta_rh`, `beta_vp` | Humidity / vapour-pressure sensitivity to NDVI deviation |
| `DOWNSCALE.gamma_wind` | Wind reduction per unit NDVI (surface roughness) |
| `GEE_FLAGS` | Must match how the export was produced: NDVI scaling, cloud mask, gap fill, MRT mode (`air` or `ssrd`) |
| `LST_MIN_C`, `LST_MAX_C` | Plausible LST range; below 15 °C is a cloud-top artefact in Chennai |
| `LST_BELOW_AIR_C` | Daytime LST this far below air temperature is treated as cloud |
| `MIN_VALID_FRAC` | A zone-scene is used only if at least this share of its points is clear |
| `GRID_STEP_DAYS` | Combined Landsat 8 + 9 revisit; the 16-day era becomes gap slots |
| `TEST_FRAC` | Final share of the timeline replayed as real time |

| Environment variable | Overrides |
|---|---|
| `UTCI_MODE` | `full` or `quick` |
| `UTCI_DATA_PATH`, `UTCI_FETCHED_PATH`, `UTCI_OUT_DIR` | File locations |
| `UTCI_GEE_FETCH`, `UTCI_GEE_DRY_RUN` | `1` / `0` |
| `GEE_PROJECT` | Earth Engine Cloud project id |
| `GEE_SERVICE_ACCOUNT_KEY` | Service-account JSON for non-interactive login (GitHub Actions) |

</details>

## 1. Load the export and any fetched scenes

In [ ]:
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_PATH}' not found. Choose the CSV to upload:")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"'{DATA_PATH}' not found. Put it next to the notebook or set DATA_PATH.")

def read_csv(path):
    d = pd.read_csv(path, encoding="utf-8-sig")
    d.columns = [c.strip() for c in d.columns]
    return d

def parse_dates(s):
    s = s.astype(str).str.strip()
    for fmt in ("%d-%m-%Y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return pd.to_datetime(s, format=fmt).dt.normalize()
        except Exception:
            pass
    return pd.to_datetime(s, dayfirst=True).dt.normalize()

def load_export():
    raw = read_csv(DATA_PATH)
    if DATE_COL not in raw.columns:
        raise ValueError(f"'{DATE_COL}' not found. Columns: {list(raw.columns)}")
    raw["Date"] = parse_dates(raw[DATE_COL])
    raw["Zone"] = raw["Point_ID"].str[:2]
    raw["Scene"] = raw["Date"].dt.strftime("D%Y-%m-%d")
    raw["Source"] = "hist"
    return raw

def read_fetched():
    if not os.path.exists(FETCHED_PATH):
        return None
    return pd.read_csv(FETCHED_PATH, parse_dates=["Date"], dtype={"Scene": str})

def load_store():
    raw = load_export()
    new = read_fetched()
    if new is not None and len(new):
        new = new[new["Date"] > raw["Date"].max()]
        raw = pd.concat([raw, new], ignore_index=True)
    return raw.sort_values("Date", kind="stable").reset_index(drop=True)

def append_fetched(rows):
    old = read_fetched()
    merged = rows if old is None else pd.concat([old, rows], ignore_index=True)
    merged = merged.drop_duplicates(["Scene", "Point_ID"], keep="last").sort_values("Date", kind="stable")
    merged.to_csv(FETCHED_PATH, index=False)

raw = load_store()
ZONES = [z for z in ZONE_NAMES if z in set(raw["Zone"])]
ZONE_NAMES.update({z: z for z in set(raw["Zone"]) - set(ZONE_NAMES)})
print(f"rows {len(raw):,} | points {raw.Point_ID.nunique()} | scenes {raw.Scene.nunique()} "
      f"| {raw.Date.min().date()} to {raw.Date.max().date()} | fetched scenes {raw.loc[raw.Source != 'hist', 'Scene'].nunique()}")

## 1b. Live Earth Engine fetch

Lists Landsat 8/9 Collection 2 L2 scenes over campus after the last stored date, then extracts LST, NDVI and the ERA5-Land hour at the 500 stored points (30 m, `first` reducer). Masked pixels are dropped, as in the export.

- **Login:** interactive (`ee.Authenticate`) locally or in Colab; a service account via `GEE_SERVICE_ACCOUNT_KEY` in GitHub Actions.
- **Latency:** Landsat L2 appears about 1–2 weeks after the overpass, and ERA5-Land runs 5–7 days behind. A scene without its ERA5-Land hour stops the fetch and is retried next run.
- Test with `GEE_DRY_RUN = True` first; it lists scenes without saving anything.

In [ ]:
def gee_init():
    global ee
    ensure_pkg("import ee", "earthengine-api")
    import ee as _ee
    ee = _ee
    key = ENV("GEE_SERVICE_ACCOUNT_KEY")
    if key:
        creds = ee.ServiceAccountCredentials(json.loads(key)["client_email"], key_data=key)
        ee.Initialize(creds, project=GEE_PROJECT)
        return
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        if IN_CI:
            raise
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)

def mask_landsat(img, flags):
    qa = img.select("QA_PIXEL")
    bits = 0b11110 if flags.get("extra_cloud_mask") else (1 << 3)   # dilated cloud, cirrus, cloud, shadow
    return img.updateMask(qa.bitwiseAnd(bits).eq(0)).updateMask(img.select("QA_RADSAT").eq(0))

def add_lst_ndvi(img, flags):
    lst = img.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("LST")
    bands = img.select(["SR_B5", "SR_B4"])
    if flags.get("fix_ndvi_scaling"):
        bands = bands.multiply(0.0000275).add(-0.2)
    return img.addBands([lst, bands.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")])

def era5_bands(img, flags):
    t2m = img.select("temperature_2m").subtract(273.15).rename("AirTemp_2m")
    d2m = img.select("dewpoint_temperature_2m").subtract(273.15).rename("DewPoint")
    wind = img.select("u_component_of_wind_10m").hypot(img.select("v_component_of_wind_10m")).rename("WindSpeed_10m")
    vp = img.expression("0.61078 * exp((17.27 * T) / (T + 237.3))", {"T": d2m}).rename("VaporPressure_kPa")
    rh = img.expression("100 * (exp((17.27 * Td) / (Td + 237.3)) / exp((17.27 * Ta) / (Ta + 237.3)))",
                        {"Td": d2m, "Ta": t2m}).clamp(0, 100).rename("RelHumidity")
    if flags.get("mrt_mode", "air") == "ssrd":
        ssrd = img.select("surface_solar_radiation_downwards").divide(3600)
        mrt = img.expression("Ta + 0.01 * SSRD", {"Ta": t2m, "SSRD": ssrd}).rename("MeanRadiantTemp")
    else:
        mrt = t2m.rename("MeanRadiantTemp")
    return img.addBands([t2m, d2m, wind, vp, rh, mrt])

def gee_list_new_images(start, end, bbox):
    roi = ee.Geometry.Rectangle(bbox)
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(roi).filterDate(start, end).sort("system:time_start"))
    info = lambda im: ee.Feature(None, {"asset_id": ee.Image(im).id(), "date": ee.Image(im).date().format("YYYY-MM-dd")})
    return [f["properties"] for f in ee.FeatureCollection(col.map(info)).getInfo()["features"]]

def gee_extract_scene(asset_id, points, flags):
    img = add_lst_ndvi(mask_landsat(ee.Image(asset_id), flags), flags)
    era = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").filterDate(img.date().advance(-1, "hour"), img.date().advance(1, "hour"))
    if era.size().getInfo() == 0:
        return None
    era1 = era5_bands(era.first(), flags)
    lst, ndvi = img.select("LST"), img.select("NDVI")
    if flags.get("fill_masked"):
        lst, ndvi = lst.unmask(era1.select("AirTemp_2m").add(1.33)), ndvi.unmask(0.45)
    comb = lst.rename("LST").addBands(ndvi.rename("NDVI")).addBands(
        era1.select(["AirTemp_2m", "MeanRadiantTemp", "RelHumidity", "VaporPressure_kPa", "WindSpeed_10m"]))
    if flags.get("fill_masked"):
        comb = comb.addBands(img.select("LST").mask().Not().rename("LST_filled"))
    pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {"Point_ID": r.Point_ID})
                                for r in points.rename(columns={"Longitude (°E)": "lon", "Latitude (°N)": "lat"}).itertuples()])
    feats = comb.reduceRegions(collection=pts, reducer=ee.Reducer.first(), scale=30).getInfo()["features"]
    return pd.DataFrame([f["properties"] for f in feats])

In [ ]:
GEE_FETCH_REPORT = []

def fetch_new_scenes_from_gee():
    store = load_store()
    last = store["Date"].max()
    start = (last + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    end = (pd.Timestamp.now(tz="UTC").tz_localize(None).normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    pts = store.drop_duplicates("Point_ID")[["Point_ID", "Latitude (°N)", "Longitude (°E)", "Zone"]].reset_index(drop=True)
    bbox = [pts["Longitude (°E)"].min() - 0.002, pts["Latitude (°N)"].min() - 0.002,
            pts["Longitude (°E)"].max() + 0.002, pts["Latitude (°N)"].max() + 0.002]
    gee_init()
    imgs = gee_list_new_images(start, end, bbox)
    print(f"Landsat 8/9 scenes over campus after {last.date()}: {len(imgs)}")
    for im in imgs[:12]:
        print("   ", im["date"], im["asset_id"].split("/")[-1])
    if GEE_DRY_RUN or not imgs:
        return 0
    have, added = set(store["Scene"].astype(str)), []
    for im in imgs[:GEE_MAX_NEW_SCENES]:
        sid = im["asset_id"].split("/")[-1]
        if sid in have:
            continue
        ex = gee_extract_scene(im["asset_id"], pts, GEE_FLAGS)
        if ex is None:
            print(f"   {im['date']}: ERA5-Land not available yet; stopping, will retry next run")
            break
        if not GEE_FLAGS.get("fill_masked"):
            ex = ex.dropna(subset=["LST", "NDVI"])
        GEE_FETCH_REPORT.append({"fetched_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                                 "date": im["date"], "scene": sid, "points": int(len(ex)),
                                 "lst_min": float(ex["LST"].min()) if len(ex) else np.nan,
                                 "lst_max": float(ex["LST"].max()) if len(ex) else np.nan,
                                 "flags": json.dumps(GEE_FLAGS)})
        if ex.empty:
            print(f"   {im['date']}  {sid}: fully cloudy")
            continue
        sc = pts.merge(ex, on="Point_ID", how="inner")
        sc["Date"], sc["Scene"], sc["Source"] = pd.Timestamp(im["date"]), sid, "gee"
        added.append(sc)
        print(f"   + {im['date']}  {sid}  clear points {len(sc)}/{len(pts)}")
    if GEE_FETCH_REPORT:
        lg = pd.DataFrame(GEE_FETCH_REPORT)
        if os.path.exists(GEE_LOG_PATH):
            lg = pd.concat([pd.read_csv(GEE_LOG_PATH), lg])
        lg.to_csv(GEE_LOG_PATH, index=False)
    if added:
        append_fetched(pd.concat(added, ignore_index=True))
    return len(added)

if RUN_GEE_FETCH:
    if fetch_new_scenes_from_gee():
        raw = load_store()
        print(f"store now has {raw['Scene'].nunique()} scenes, last date {raw['Date'].max().date()}")
else:
    print("Earth Engine fetch is off.")

## 1c. Downscaling (TVX)

Per date: drop points with LST below 15 °C, take the campus 99th-percentile NDVI as the canopy anchor, then regress LST on NDVI per zone-date.

| Output | Formula |
|---|---|
| `Local_AirTemp_2m` | `T_amb + α (LST − T_amb)`, where `T_amb` is the regression at the canopy NDVI |
| `Local_MRT` | `MRT + k (LST − mean LST)` |
| `Local_RelHumidity` | `RH + β (NDVI − mean NDVI)` |
| `Local_WindSpeed_10m` | `max(0.5, v (1 − γ NDVI))` |

A zone-date with no LST or NDVI variation keeps the ERA5 values.

In [ ]:
BASE_COLS = ["LST", "NDVI", "AirTemp_2m", "MeanRadiantTemp", "RelHumidity", "VaporPressure_kPa", "WindSpeed_10m"]
ERA_COLS = ["AirTemp_2m", "MeanRadiantTemp", "RelHumidity", "VaporPressure_kPa", "WindSpeed_10m"]
LOCAL_COLS = ["Local_AirTemp_2m", "Local_MRT", "Local_RelHumidity", "Local_VaporPressure_kPa", "Local_WindSpeed_10m"]

def qc_valid(df):
    filled = (df["LST_filled"].fillna(0).astype(float) > 0.5) if "LST_filled" in df.columns else pd.Series(False, index=df.index)
    lst_ok = df["LST"].between(LST_MIN_C, LST_MAX_C) & (df["LST"] >= df["AirTemp_2m"] - LST_BELOW_AIR_C)
    return lst_ok & ~filled & df[BASE_COLS].notna().all(axis=1)

def tvx_zone_date(g, canopy_ndvi, p):
    out = pd.DataFrame(g[ERA_COLS].values, columns=LOCAL_COLS, index=g.index).astype(float)
    if pd.isna(canopy_ndvi) or g["NDVI"].nunique() <= 1 or g["LST"].nunique() <= 1:
        return out
    slope, intercept, *_ = linregress(g["NDVI"], g["LST"])
    t_amb = slope * canopy_ndvi + intercept
    ndvi_dev = g["NDVI"] - g["NDVI"].mean()
    out["Local_AirTemp_2m"] = t_amb + p["alpha_air"] * (g["LST"] - t_amb)
    out["Local_MRT"] = g["MeanRadiantTemp"] + p["k_mrt"] * (g["LST"] - g["LST"].mean())
    out["Local_RelHumidity"] = (g["RelHumidity"] + p["beta_rh"] * ndvi_dev).clip(0, 100)
    out["Local_VaporPressure_kPa"] = g["VaporPressure_kPa"] + p["beta_vp"] * ndvi_dev
    out["Local_WindSpeed_10m"] = (g["WindSpeed_10m"] * (1 - p["gamma_wind"] * g["NDVI"])).clip(lower=0.5)
    return out

def downscale(df):
    df = df.drop(columns=[c for c in LOCAL_COLS if c in df.columns]).reset_index(drop=True)
    df[LOCAL_COLS] = df[ERA_COLS].values
    clear = df[df["LST"].ge(LST_MIN_C) & df[["LST", "NDVI"]].notna().all(axis=1)]
    canopy = clear.groupby("Date")["NDVI"].quantile(0.99).to_dict()
    for (_, date), g in clear.groupby(["Zone", "Date"], sort=False):
        df.loc[g.index, LOCAL_COLS] = tvx_zone_date(g, canopy.get(date, np.nan), DOWNSCALE).values
    return df

t_ds = time.time()
raw = downscale(raw)
N_DOWNSCALED = int(raw["Date"].nunique())
print(f"downscaled {N_DOWNSCALED} dates in {time.time() - t_ds:.0f}s")

ok = qc_valid(raw)
zone_inputs = raw[ok].groupby("Zone")[["LST", "NDVI", "Local_MRT", "Local_RelHumidity", "Local_WindSpeed_10m"]].mean().round(2)
zone_inputs.insert(0, "clear_points", raw[ok].groupby("Zone").size())
display(zone_inputs)

## 2. Cloud QC

In [ ]:
def clean(raw):
    df = raw.copy()
    g = df.groupby("Scene")
    gap = (g["LST"].transform("std").fillna(0) < GAPFILL_STD_TOL) & (g["NDVI"].transform("std").fillna(0) < GAPFILL_STD_TOL)
    df["gapfilled_scene"] = gap
    df["point_valid"] = ~gap & qc_valid(df) & df[list(UTCI_INPUTS.values())].notna().all(axis=1)
    expected = df.groupby("Zone")["Point_ID"].nunique().reindex(ZONES)
    n_valid = (df[df["point_valid"]].groupby(["Scene", "Zone"]).size().unstack(fill_value=0)
               .reindex(index=df["Scene"].unique(), columns=ZONES, fill_value=0))
    frac = n_valid.div(expected, axis=1)
    frac.index.name, frac.columns.name = "Scene", "Zone"
    df = df.join(frac.stack().rename("valid_frac"), on=["Scene", "Zone"])
    df["use"] = df["point_valid"] & (df["valid_frac"] >= MIN_VALID_FRAC)
    summ = frac.copy()
    summ.insert(0, "points_present", df.groupby("Scene").size())
    summ.insert(0, "gapfilled", df.groupby("Scene")["gapfilled_scene"].first())
    summ.insert(0, "Date", df.groupby("Scene")["Date"].first())
    return df, summ.sort_values("Date", kind="stable")

df, clean_summary = clean(raw)
clean_summary.to_csv(out("cleaning_summary_by_scene.csv"))
CLEAN_STATS = {
    "n_scenes": int(len(clean_summary)),
    "n_gapfilled_or_single_point_scenes": int(clean_summary["gapfilled"].sum()),
    "points_valid_frac_of_rows": round(float(df["point_valid"].mean()), 4),
    "median_points_per_scene": float(clean_summary["points_present"].median()),
    "zone_scenes_usable": {z: int((clean_summary[z] >= MIN_VALID_FRAC).sum()) for z in ZONES},
}
print(json.dumps(CLEAN_STATS, indent=1))

from matplotlib.colors import LinearSegmentedColormap
clear_cmap = LinearSegmentedColormap.from_list("clear", ["#F3EEE7", PALETTE["gold"], PALETTE["maroon"]])
fig, ax = plt.subplots(figsize=(14, 2.6))
im = ax.imshow(clean_summary[ZONES].T.values, aspect="auto", cmap=clear_cmap, vmin=0, vmax=1, interpolation="nearest")
ax.set_yticks(range(len(ZONES)), ZONES); ax.grid(False)
ax.set_xlabel("scene (chronological)")
ax.set_title("Share of each zone's points that are clear, per scene")
plt.colorbar(im, ax=ax, fraction=0.02)
plt.savefig(out("fig01_cleaning.png")); plt.show()

## 3. UTCI per point, then zone means

In [ ]:
def compute_utci(tdb, tr, v, rh):
    res = ptc_utci(tdb=np.asarray(tdb, float), tr=np.asarray(tr, float), v=np.asarray(v, float), rh=np.asarray(rh, float))
    val = getattr(res, "utci", res)
    return np.asarray(val["utci"] if isinstance(val, dict) else val, float)

I = UTCI_INPUTS
MRT_EQUALS_AIR = bool((raw["MeanRadiantTemp"] - raw["AirTemp_2m"]).abs().max() < 1e-6)
if MRT_EQUALS_AIR:
    print("Note: MeanRadiantTemp equals AirTemp_2m in every row, so UTCI has no solar term (only the LST deviation adds radiant load).")

df["UTCI"] = compute_utci(df[I["tdb"]], df[I["tr"]], df[I["v"]], df[I["rh"]])
n_out_of_range = int((df["use"] & df["UTCI"].isna()).sum())
df.loc[~df["use"], "UTCI"] = np.nan
print("clear points outside the UTCI input range:", n_out_of_range)

variants = {
    "chosen": I,
    "all_local": {"tdb": "Local_AirTemp_2m", "tr": "Local_MRT", "v": "Local_WindSpeed_10m", "rh": "Local_RelHumidity"},
    "all_era5": {"tdb": "AirTemp_2m", "tr": "MeanRadiantTemp", "v": "WindSpeed_10m", "rh": "RelHumidity"},
}
u = df[df["use"]]
utci_input_comparison = pd.DataFrame({
    name: pd.Series(compute_utci(u[c["tdb"]], u[c["tr"]], u[c["v"]], u[c["rh"]]), index=u.index)
            .groupby(u["Zone"]).agg(["mean", "std", "min", "max"]).stack()
    for name, c in variants.items()}).round(3)
utci_input_comparison.to_csv(out("utci_input_comparison.csv"))
display(utci_input_comparison.unstack().xs("mean", axis=1, level=1))

zone_scene = df[df["use"]].groupby(["Date", "Zone"])["UTCI"].mean().unstack().reindex(columns=ZONES)
zone_scene.to_csv(out("zone_utci_by_scene.csv"))

## 4. Regular 8-day grid

In [ ]:
all_dates = pd.DatetimeIndex(sorted(df["Date"].unique()))
gaps_days = np.diff(all_dates).astype("timedelta64[D]").astype(int)
STEP = int(GRID_STEP_DAYS)
T0 = all_dates[0]
n_grid = int(round((all_dates[-1] - T0).days / STEP)) + 1
GRID = pd.date_range(T0, periods=n_grid, freq=f"{STEP}D")

idx = np.round((zone_scene.index - T0).days / STEP).astype(int)
snap_off = (zone_scene.index - GRID[idx]).days
Y = zone_scene.groupby(idx).mean().reindex(range(n_grid))
Y.index = GRID
gap_by_year = pd.Series(gaps_days, index=all_dates[1:]).groupby(all_dates[1:].year).median()
GRID_STATS = {"step_days": STEP, "n_grid": n_grid,
              "scene_gap_days_counts": pd.Series(gaps_days).value_counts().head(8).to_dict(),
              "median_gap_days_by_year": {int(k): float(v) for k, v in gap_by_year.items()},
              "max_abs_snap_days": int(np.abs(snap_off).max()) if len(snap_off) else 0,
              "scenes_merged_into_same_slot": int(len(idx) - len(np.unique(idx))),
              "observed_slots_per_zone": Y.notna().sum().to_dict(),
              "zone_mean_utci": Y.mean().round(3).to_dict(),
              "zone_mean_range_C": round(float(Y.mean().max() - Y.mean().min()), 3)}
print(json.dumps(GRID_STATS, indent=1, default=int))
Y.to_csv(out("zone_utci_grid.csv"))

fig, axes = plt.subplots(len(ZONES), 1, figsize=(14, 1.9 * len(ZONES)), sharex=True)
for ax, z in zip(axes, ZONES):
    ax.plot(Y.index, Y[z].interpolate(limit_area="inside"), color=PALETTE["rule"], lw=1)
    ax.plot(Y.index, Y[z], "o", ms=2.6, color=PALETTE["maroon"])
    ax.set_ylabel(z)
axes[0].set_title("Zone-mean UTCI at overpass (°C); grey line spans cloudy or missing slots")
plt.savefig(out("fig02_zone_series.png")); plt.show()

## 5. Decomposition: climatology + trend + anomaly

Per zone: `UTCI = b0 + b1·years + Σk (ak sin + ck cos)(2πk·doy/365.25) + anomaly`.
The intercept differs per zone, so the zone offset lives in this part. The replay refits it at every step on past data only.

In [ ]:
def design(dates):
    dates = pd.DatetimeIndex(dates)
    t = (dates - T0).days.values / 365.25
    w = 2 * np.pi * dates.dayofyear.values / 365.25
    cols = [np.ones(len(dates)), t]
    for k in range(1, N_HARMONICS + 1):
        cols += [np.sin(k * w), np.cos(k * w)]
    return np.column_stack(cols)

def fit_baseline(hist):
    X = design(hist.index)
    return {z: np.linalg.lstsq(X[hist[z].notna().values], hist[z].dropna().values, rcond=None)[0] for z in hist.columns}

def baseline(dates, betas):
    X = design(dates)
    return pd.DataFrame({z: X @ b for z, b in betas.items()}, index=pd.DatetimeIndex(dates))

betas_full = fit_baseline(Y)
B_full = baseline(GRID, betas_full)
A_full = Y - B_full
DECOMP = {
    "trend_C_per_year": {z: round(float(b[1]), 3) for z, b in betas_full.items()},
    "anomaly_std": A_full.std().round(3).to_dict(),
    "raw_std": Y.std().round(3).to_dict(),
    "anomaly_lag1_autocorr": {z: round(float(A_full[z].autocorr(1)), 3) for z in ZONES},
    "anomaly_cross_zone_corr_mean": round(float(np.nanmean(A_full.corr().values[np.triu_indices(len(ZONES), 1)])), 4),
}
coef_names = ["intercept", "trend"] + [f"{s}{k}" for k in range(1, N_HARMONICS + 1) for s in ("sin", "cos")]
pd.DataFrame(betas_full, index=coef_names).T.to_csv(out("climatology_coefficients.csv"))
A_full.corr().round(3).to_csv(out("anomaly_cross_zone_corr.csv"))
print(json.dumps(DECOMP, indent=1))

z = ZONES[0]
fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
ax[0].plot(Y.index, Y[z], "o", ms=2.6, color=PALETTE["grey"], label="observed")
ax[0].plot(B_full.index, B_full[z], lw=1.8, color=PALETTE["maroon"], label="climatology + trend")
ax[0].set_ylabel("UTCI °C"); ax[0].legend(loc="upper left", ncol=2)
ax[1].bar(A_full.index, A_full[z], width=STEP * 0.8, color=np.where(A_full[z] > 0, PALETTE["maroon"], PALETTE["slate"]))
ax[1].axhline(0, color=PALETTE["ink"], lw=0.6); ax[1].set_ylabel("anomaly °C")
ax[0].set_title(f"Decomposition, {ZONE_NAMES[z]}")
plt.savefig(out("fig03_decomposition.png")); plt.show()

## 6. Models

Both predict the **anomaly** at the next grid slot from data before it; the forecast is `climatology + trend + predicted anomaly`.

| Model | Idea |
|---|---|
| `kalman` | AR(1) state-space model per zone; absorbs each new scene, skips cloudy gaps natively, gives prediction intervals |
| `lstm` | Small LSTM over the last 8 grid steps of all five zones (anomalies, observed masks, season); 3-seed ensemble with a linear skip path; full runs only |

In [ ]:
MODELS = ["kalman", "lstm"] if USE_LSTM else ["kalman"]
ZQ = {0.80: 1.2816, 0.90: 1.6449, 0.95: 1.96}[PI_LEVEL]

def predict_step(hist, target_date, kal_params=None, refit_kalman=True, use_lstm=USE_LSTM, force_lstm=False):
    betas = fit_baseline(hist)
    dates_ext = hist.index.append(pd.DatetimeIndex([target_date]))
    B = baseline(dates_ext, betas)
    A_hist = (hist - B.iloc[:-1]).values
    n, b_t = len(hist), B.iloc[-1]
    out_ = {z: {"baseline": float(b_t[z])} for z in ZONES}
    params = dict(kal_params or {})
    for zi, z in enumerate(ZONES):
        a = A_hist[:, zi]
        seen = np.where(~np.isnan(a))[0]
        out_[z].update(gap_steps=int(n - seen[-1]), last_obs_date=hist.index[seen[-1]], last_obs_anom=float(a[seen[-1]]))
        try:
            mod = SARIMAX(a, order=(1, 0, 0), trend="n")
            if refit_kalman or z not in params:
                res = mod.fit(disp=False); params[z] = np.asarray(res.params)
            else:
                res = mod.filter(params[z])
            fc = res.get_forecast(1)
            m, sd = float(fc.predicted_mean[0]), float(np.sqrt(fc.var_pred_mean[0]))
        except Exception:
            m, sd = 0.0, float(np.nanstd(a))
        out_[z].update(kalman=float(b_t[z] + m), kalman_lo=float(b_t[z] + m - ZQ * sd),
                       kalman_hi=float(b_t[z] + m + ZQ * sd), kalman_anom=m, kalman_sd=sd)
    if use_lstm:
        A_ext = np.vstack([A_hist, np.full((1, len(ZONES)), np.nan)])
        p_ls = lstm_predict(A_ext, dates_ext, n, force=force_lstm)
        for zi, z in enumerate(ZONES):
            out_[z]["lstm"] = float(b_t[z] + p_ls[zi])
    return out_, params, betas

In [ ]:
if USE_LSTM:
    ensure_pkg("import torch", "torch")
    import torch, torch.nn as nn
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    VERSIONS["torch"] = torch.__version__
    LSTM_CACHE = {"models": None, "n_fit": -10**9, "zn": None}

    class AnomLSTM(nn.Module):
        # LSTM plus a linear skip path initialised at the data's lag-1 persistence, so the LSTM learns a correction
        def __init__(self, n_in, hidden, n_out, dropout=0.2, phi0=0.5):
            super().__init__()
            self.n_out = n_out
            self.lstm = nn.LSTM(n_in, hidden, batch_first=True)
            self.drop = nn.Dropout(dropout)
            self.head = nn.Linear(hidden, n_out)
            self.skip = nn.Linear(n_out, n_out, bias=False)
            with torch.no_grad():
                self.skip.weight.copy_(phi0 * torch.eye(n_out))

        def forward(self, x):
            o, _ = self.lstm(x)
            return self.head(self.drop(o[:, -1, :])) + self.skip(x[:, -1, :self.n_out])

    def lstm_features(A, dates, scale):
        obs = ~np.isnan(A)
        w = 2 * np.pi * pd.DatetimeIndex(dates).dayofyear.values / 365.25
        return np.column_stack([np.where(obs, A / scale, 0.0), obs.astype(float), np.sin(w), np.cos(w)])

    def train_lstm(X, Yt, M, seed, epochs, init_state=None, phi0=0.5):
        torch.manual_seed(seed); np.random.seed(seed)
        model = AnomLSTM(X.shape[2], LSTM_HIDDEN, Yt.shape[1], phi0=phi0)
        if init_state is not None:
            model.load_state_dict(init_state)
        opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-3)
        Xt, Yy, Mt = (torch.tensor(a, dtype=torch.float32) for a in (X, Yt, M))
        model.train()
        for _ in range(epochs):
            perm = torch.randperm(len(Xt))
            for i in range(0, len(Xt), 32):
                b = perm[i:i + 32]
                opt.zero_grad()
                loss = (((model(Xt[b]) - Yy[b]) ** 2) * Mt[b]).sum() / Mt[b].sum().clamp(min=1.0)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        model.eval()
        return model

    def lstm_predict(A_ext, dates_ext, n, force=False):
        W, Zn = LSTM_WINDOW, A_ext.shape[1]
        scale = float(np.nanstd(A_ext[:n])) or 1.0
        F = lstm_features(A_ext, dates_ext, scale)
        window = lambda p: np.concatenate([F[p - W:p], np.tile(F[p, -2:], (W, 1))], axis=1)
        ps = [p for p in range(W, n) if (~np.isnan(A_ext[p])).any()]
        if len(ps) < 30:
            return np.zeros(Zn)
        X = np.stack([window(p) for p in ps]).astype("float32")
        Yt = np.nan_to_num(A_ext[ps] / scale).astype("float32")
        M = (~np.isnan(A_ext[ps])).astype("float32")
        a0, a1 = A_ext[:n - 1].ravel(), A_ext[1:n].ravel()
        both = ~np.isnan(a0) & ~np.isnan(a1)
        phi0 = float(np.clip(np.corrcoef(a0[both], a1[both])[0, 1], 0.0, 0.9)) if both.sum() > 10 else 0.3
        stale = LSTM_CACHE["models"] is None or LSTM_CACHE["zn"] != Zn or (n - LSTM_CACHE["n_fit"]) >= LSTM_REFIT_EVERY
        if force or stale:
            prev = None if force else LSTM_CACHE["models"]
            models = []
            for s in range(LSTM_SEEDS):
                init = prev[s].state_dict() if prev is not None else None
                models.append(train_lstm(X, Yt, M, seed=s, epochs=LSTM_WARM_EPOCHS if init else LSTM_EPOCHS,
                                         init_state=init, phi0=phi0))
            LSTM_CACHE.update(models=models, n_fit=n, zn=Zn)
        xq = torch.tensor(window(n)[None], dtype=torch.float32)
        with torch.no_grad():
            return np.mean([m(xq).numpy()[0] for m in LSTM_CACHE["models"]], axis=0) * scale
else:
    print("Quick mode: LSTM skipped.")

## 7–8. Rolling-origin replay and metrics *(full run)*

For every grid slot in the last `TEST_FRAC` of the timeline: fit on data before it, predict it, move on. This is what the live system does each time a scene arrives. Quick runs reuse the metrics of the last full run.

In [ ]:
def run_replay():
    test_start = int(len(GRID) * (1 - TEST_FRAC))
    recs, kal_params = [], {}
    LSTM_CACHE.update(models=None, n_fit=-10**9, zn=None)
    t0 = time.time()
    for step_i, i in enumerate(range(test_start, len(GRID))):
        if Y.iloc[i].isna().all():
            continue
        refit = (step_i % KALMAN_REFIT_EVERY == 0) or not kal_params
        o, kal_params, _ = predict_step(Y.iloc[:i], GRID[i], kal_params, refit_kalman=refit)
        for z in ZONES:
            if np.isnan(Y.iloc[i][z]):
                continue
            recs.append({"date": GRID[i], "zone": z, "actual": float(Y.iloc[i][z]), "gap_steps": o[z]["gap_steps"],
                         "baseline": o[z]["baseline"], "kalman_lo": o[z]["kalman_lo"], "kalman_hi": o[z]["kalman_hi"],
                         **{m: o[z][m] for m in MODELS}})
    print(f"replay: {len(recs)} zone-predictions in {time.time() - t0:.0f}s")
    P = pd.DataFrame(recs)
    P["month"] = P["date"].dt.month
    P["gap_bucket"] = np.select([P["gap_steps"] <= 1, P["gap_steps"] == 2], ["1 step", "2 steps"], "3+ steps")
    P.to_csv(out("rolling_predictions.csv"), index=False)
    return P, test_start

def rmse_table(P, by):
    return P.groupby(by).apply(lambda g: pd.Series({m: np.sqrt(np.mean((g[m] - g["actual"]) ** 2)) for m in MODELS} | {"n": len(g)})).round(3)

def evaluate(P, test_start):
    rows = {m: {"MAE": (P[m] - P["actual"]).abs().mean(), "RMSE": np.sqrt(((P[m] - P["actual"]) ** 2).mean()),
                "bias": (P[m] - P["actual"]).mean()} for m in MODELS}
    tbl = pd.DataFrame(rows).T
    tbl["skill_vs_kalman"] = 1 - tbl["RMSE"] / tbl.loc["kalman", "RMSE"]
    tbl = tbl.sort_values("RMSE").round(3)
    tbl.to_csv(out("metrics_overall.csv"))
    for by, name in [("zone", "zone"), ("gap_bucket", "gap"), ("month", "month")]:
        rmse_table(P, by).to_csv(out(f"metrics_rmse_by_{name}.csv"))
    inside = (P["actual"] >= P["kalman_lo"]) & (P["actual"] <= P["kalman_hi"])
    ev = {"overall": tbl, "best": tbl.index[0],
          "test_window": {"start": str(GRID[test_start].date()), "end": str(GRID[-1].date()), "n_predictions": int(len(P))},
          "kalman_interval": {"level": PI_LEVEL, "empirical_coverage": round(float(inside.mean()), 3),
                              "mean_width": round(float((P["kalman_hi"] - P["kalman_lo"]).mean()), 3)},
          "last_full_run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    if "lstm" in P:
        ae_k, ae_l = (P["kalman"] - P["actual"]).abs(), (P["lstm"] - P["actual"]).abs()
        ev["head_to_head"] = {"lstm_better_share": round(float((ae_l < ae_k).mean()), 3),
                              "mean_abs_err_kalman": round(float(ae_k.mean()), 3), "mean_abs_err_lstm": round(float(ae_l.mean()), 3),
                              "mean_paired_diff_kalman_minus_lstm": round(float((ae_k - ae_l).mean()), 3),
                              "paired_diff_std_err": round(float((ae_k - ae_l).std() / np.sqrt(len(P))), 3)}
    return ev

def plot_replay(P):
    fig, axes = plt.subplots(len(ZONES), 1, figsize=(14, 2.1 * len(ZONES)), sharex=True)
    for ax, z in zip(axes, ZONES):
        q = P[P.zone == z]
        ax.fill_between(q.date, q.kalman_lo, q.kalman_hi, color=PALETTE["maroon"], alpha=0.12, lw=0, label=f"Kalman {int(PI_LEVEL * 100)}% interval")
        ax.plot(q.date, q.kalman, color=PALETTE["maroon"], lw=1.2, label="Kalman")
        if "lstm" in q:
            ax.plot(q.date, q.lstm, color=PALETTE["slate"], lw=1, alpha=0.9, label="LSTM")
        ax.plot(q.date, q.actual, "o", ms=2.6, color=PALETTE["ink"], label="actual")
        ax.set_ylabel(z)
    axes[0].legend(ncol=4, loc="upper left", fontsize=8)
    axes[0].set_title("Rolling replay: one-step-ahead predictions (°C)")
    plt.savefig(out("fig04_replay_predictions.png")); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
    for m, c in zip(MODELS, [PALETTE["maroon"], PALETTE["slate"]]):
        e = P[m] - P["actual"]
        ax[0].plot(e.groupby(P["date"]).mean(), lw=1, color=c, label=m)
        ax[1].hist(e, bins=30, alpha=0.45, color=c, label=m)
    ax[0].axhline(0, color=PALETTE["ink"], lw=0.6); ax[0].set_title("Error over time (predicted − actual, zone mean)"); ax[0].legend()
    ax[1].set_title("Error distribution"); ax[1].legend()
    plt.savefig(out("fig05_errors.png")); plt.show()

def load_previous_eval():
    path = out("run_summary.json")
    if not (os.path.exists(path) and os.path.exists(out("metrics_overall.csv"))):
        raise FileNotFoundError("Quick mode needs the outputs of a full run in OUT_DIR. Run once with UTCI_MODE=full.")
    prev = json.load(open(path))
    return {"overall": pd.read_csv(out("metrics_overall.csv"), index_col=0), "best": prev.get("best_model_by_rmse", "kalman"),
            "test_window": prev.get("test_window"), "kalman_interval": prev.get("kalman_interval"),
            "head_to_head": prev.get("head_to_head"),
            "last_full_run_utc": prev.get("last_full_run_utc", prev.get("run_utc"))}

if FULL_RUN:
    P, TEST_START = run_replay()
    EVAL = evaluate(P, TEST_START)
    display(EVAL["overall"])
    print("Kalman interval:", EVAL["kalman_interval"])
    if "head_to_head" in EVAL:
        print("Kalman vs LSTM:", EVAL["head_to_head"])
    plot_replay(P)
else:
    EVAL = load_previous_eval()
    print(f"Quick mode: metrics from the full run of {EVAL['last_full_run_utc']}")
    display(EVAL["overall"])

## 9. Live forecast, log and scorecard

- The next grid slot comes from `FORECAST_MODEL`, fitted on all data.
- **Daily fill (Kalman):** `climatology(day) + trend + a_last · φ^k`, where `k` is days since the last scene divided by 8, with interval `z·σ·√((1 − φ^2k)/(1 − φ²))`. At the next slot this equals the Kalman forecast exactly, and it is the same formula the dashboard uses.
- Values are 10:30 AM overpass conditions. The in-between days cannot be validated; there is no daily ground truth.
- Every forecast is appended to `forecast_log.csv` and scored once its scene arrives.

In [ ]:
def live_forecast():
    next_date = GRID[-1] + pd.Timedelta(days=STEP)
    o, params, betas = predict_step(Y, next_date, force_lstm=True)
    days = pd.date_range(GRID[-1] + pd.Timedelta(days=1), next_date, freq="D")
    rmse_model = float(EVAL["overall"].loc[FORECAST_MODEL, "RMSE"]) if FORECAST_MODEL in EVAL["overall"].index else np.nan
    rows = []
    for z in ZONES:
        phi, s2 = max(float(params[z][0]), 0.0), float(params[z][1])
        d_last, a_last = o[z]["last_obs_date"], o[z]["last_obs_anom"]
        a_next = o[z][FORECAST_MODEL] - o[z]["baseline"]
        base = baseline(days, {z: betas[z]})[z]
        for d in days:
            k = (d - d_last).days / STEP
            if FORECAST_MODEL == "kalman":
                val = base[d] + a_last * phi ** k
                hw = ZQ * np.sqrt(s2 * (1 - phi ** (2 * k)) / (1 - phi ** 2))
            else:
                f = (d - d_last).days / (next_date - d_last).days
                val, hw = base[d] + a_last + (a_next - a_last) * f, ZQ * rmse_model * np.sqrt(f)
            rows.append({"zone": z, "zone_name": ZONE_NAMES[z], "date": d.date(),
                         "type": "forecast" if d == next_date else "interpolated",
                         "utci_1030": round(val, 2), "lo": round(val - hw, 2), "hi": round(val + hw, 2),
                         "model": FORECAST_MODEL, "last_obs_date": d_last.date(), "climatology": round(base[d], 2)})
    kp = pd.DataFrame({z: {"ar.L1": float(params[z][0]), "sigma2": float(params[z][1])} for z in ZONES}).T
    return pd.DataFrame(rows), next_date, o, kp

FC, NEXT_DATE, FC_RAW, KALMAN_PARAMS = live_forecast()
FC.to_csv(out("forecast_next.csv"), index=False)
KALMAN_PARAMS.to_csv(out("kalman_params.csv"))
display(FC[FC.type == "forecast"].set_index("zone")[["date", "utci_1030", "lo", "hi", "last_obs_date"]])

issued = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
log_new = pd.DataFrame([{"issued_utc": issued, "data_until": GRID[-1].date(), "target_date": NEXT_DATE.date(), "zone": z,
                         "model": m, "pred": FC_RAW[z][m],
                         "lo": FC_RAW[z]["kalman_lo"] if m == "kalman" else np.nan,
                         "hi": FC_RAW[z]["kalman_hi"] if m == "kalman" else np.nan}
                        for z in ZONES for m in MODELS])
log = pd.concat([pd.read_csv(LOG_PATH), log_new]) if os.path.exists(LOG_PATH) else log_new
for c in ("target_date", "data_until"):
    log[c] = pd.to_datetime(log[c]).dt.date.astype(str)
log = log.drop(columns=["placeholder_dates"], errors="ignore").drop_duplicates(["data_until", "target_date", "zone", "model"], keep="last")
log.to_csv(LOG_PATH, index=False)

act = Y.stack().dropna().rename("actual").reset_index()
act.columns = ["target_date", "zone", "actual"]
act["target_date"] = act["target_date"].dt.date.astype(str)
score = log.merge(act, on=["target_date", "zone"], how="inner")
score["error"] = score["pred"] - score["actual"]
score["inside_interval"] = (score["actual"] >= score["lo"]) & (score["actual"] <= score["hi"])
score.to_csv(out("realtime_scorecard.csv"), index=False)
print(f"live forecasts scored so far: {len(score)} rows")
if len(score):
    display(score.groupby("model")["error"].agg(live_RMSE=lambda e: np.sqrt(np.mean(e ** 2)), n="size").round(3))

fig, axes = plt.subplots(len(ZONES), 1, figsize=(12, 2 * len(ZONES)), sharex=True)
recent = Y.iloc[-20:]
for ax, z in zip(axes, ZONES):
    q = FC[FC.zone == z]; dd = pd.to_datetime(q.date)
    ax.fill_between(dd, q.lo, q.hi, color=PALETTE["maroon"], alpha=0.12, lw=0)
    ax.plot(dd, q.utci_1030, color=PALETTE["maroon"], lw=1.4)
    ax.plot(recent.index, recent[z], "o", ms=3, color=PALETTE["ink"])
    f = q[q.type == "forecast"]
    ax.plot(pd.to_datetime(f.date), f.utci_1030, "D", ms=6, color=PALETTE["gold"])
    ax.set_ylabel(z)
axes[0].set_title(f"Next-scene forecast for {NEXT_DATE.date()} (diamond) and daily fill, {FORECAST_MODEL}")
plt.savefig(out("fig06_next_forecast.png")); plt.show()

## 10. Manual ingest

Fallback when Earth Engine is unavailable: export a scene yourself with the same columns as the raw export, add it here, then re-run from section 1.

In [ ]:
def ingest_new_scene(path, scene_date=None):
    new = read_csv(path)
    if DATE_COL in new.columns:
        new["Date"] = parse_dates(new[DATE_COL])
    elif scene_date is not None:
        new["Date"] = pd.Timestamp(scene_date).normalize()
    else:
        raise ValueError("file has no date column; pass scene_date='YYYY-MM-DD'")
    new["Zone"] = new["Point_ID"].str[:2]
    new["Scene"], new["Source"] = "M" + new["Date"].dt.strftime("%Y%m%d"), "manual"
    append_fetched(new)
    print(f"ingested {len(new)} rows into {FETCHED_PATH}. Re-run from section 1.")

# ingest_new_scene("new_scene_2026-11-05.csv")

## 11. Run summary

In [ ]:
SUMMARY = {
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "mode": "full" if FULL_RUN else "quick",
    "last_full_run_utc": EVAL["last_full_run_utc"],
    "versions": VERSIONS,
    "config": {"UTCI_INPUTS": UTCI_INPUTS, "DOWNSCALE": DOWNSCALE, "GEE_FLAGS": GEE_FLAGS, "mrt_equals_air_temp": MRT_EQUALS_AIR,
               "LST_MIN_C": LST_MIN_C, "LST_MAX_C": LST_MAX_C, "LST_BELOW_AIR_C": LST_BELOW_AIR_C,
               "MIN_VALID_FRAC": MIN_VALID_FRAC, "GRID_STEP_DAYS": GRID_STEP_DAYS, "N_HARMONICS": N_HARMONICS,
               "TEST_FRAC": TEST_FRAC, "PI_LEVEL": PI_LEVEL, "FORECAST_MODEL": FORECAST_MODEL},
    "data": {"rows": int(len(raw)), "scenes": int(raw["Scene"].nunique()),
             "scenes_fetched": int(raw.loc[raw.Source != "hist", "Scene"].nunique()),
             "first_date": str(raw.Date.min().date()), "last_date": str(raw.Date.max().date()),
             "last_usable_scene": str(max(Y[z].dropna().index[-1] for z in ZONES).date())},
    "gee_fetch_this_run": GEE_FETCH_REPORT, "scenes_downscaled": N_DOWNSCALED,
    "cleaning": CLEAN_STATS, "utci_out_of_range_points": n_out_of_range, "grid": GRID_STATS, "decomposition": DECOMP,
    "test_window": EVAL["test_window"], "kalman_interval": EVAL["kalman_interval"], "head_to_head": EVAL.get("head_to_head"),
    "best_model_by_rmse": EVAL["best"], "next_forecast_date": str(NEXT_DATE.date()),
    "live_scored_rows": int(len(score)),
    "runtime_s": round(time.time() - T_START, 1),
}
with open(out("run_summary.json"), "w") as f:
    json.dump(SUMMARY, f, indent=1, default=str)

if not IN_CI:
    zip_path = "outputs_for_review.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fn in sorted(os.listdir(OUT_DIR)):
            zf.write(os.path.join(OUT_DIR, fn), arcname=fn)
    print("bundle:", os.path.abspath(zip_path))
print(f"done in {SUMMARY['runtime_s']}s | next forecast {SUMMARY['next_forecast_date']} | last usable scene {SUMMARY['data']['last_usable_scene']}")